<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/ADPs/Practica/Practica_2_RAG_%D0%B0%D1%81%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%82_%D0%BF%D0%BE_%D0%BF%D1%81%D0%B8%D1%85%D0%BE%D0%BB%D0%BE%D0%B3%D0%B8%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# RAG-ассистент по психологии

## Что мы делаем и зачем

В первом проекте модель отвечала, полагаясь только на свои «знания». Теперь мы научим её **сначала находить нужные фрагменты в вашей книге или лекциях, а потом отвечать, глядя на них**. Это называется RAG (Retrieval-Augmented Generation).

Представьте, что вы готовитесь к экзамену: вместо того чтобы отвечать по памяти, вы открываете учебник, находите нужный абзац и формулируете ответ, опираясь на него. Именно так будет работать наша программа.

**Что должно быть готово до начала:** Ollama, хотя бы одна чат-модель (например, `phi3:mini` или `qwen2.5:3b`), библиотека `requests`. Всё это вы уже сделали в первом проекте.

---

## Шаг 1. Создайте папку проекта в VS Code (подробно, как в учебнике)

1. Откройте **Проводник Windows** (значок папки на панели задач).
2. Перейдите в папку `Документы` (или в любое удобное место, например `C:\Users\ВашеИмя\Documents`).
3. Создайте новую папку. Назовите её без пробелов и русских букв: `rag_psychology`. Это будет **корневая папка проекта**.
4. Внутри `rag_psychology` создайте ещё две папки:
   - `data` — здесь будет лежать текст книги или лекций;
   - `code` — здесь будут лежать Python-скрипты.

Теперь структура выглядит так:
```
rag_psychology/
├── data/
└── code/
```

5. Откройте **VS Code**.
6. В верхнем меню выберите **File → Open Folder…** (Файл → Открыть папку).
7. Найдите папку `rag_psychology` и нажмите «Выбрать папку».
8. Если VS Code спросит, доверяете ли вы авторам файлов, нажмите **Yes, I trust the authors**.

В левой панели (Explorer) теперь видна ваша папка и две подпапки внутри неё. Это и есть проект.

---

## Шаг 2. Положите книгу (или лекции) в папку `data`

Для учебного RAG не нужен огромный учебник. Достаточно **3–10 страниц связного текста по одной теме**. Чем чётче тема, тем проще проверить, хорошо ли работает поиск.

**Варианты текстов:**
- Глава из учебника по общей психологии (например, раздел «Эмоции» или «Когнитивные искажения»).
- Конспект лекций, который вы написали сами.
- Открытый учебник: например, **OpenStax Psychology 2e** (лицензия CC BY, доступен на английском). Если хотите русский текст, можно взять открытые методички для непрофильных специальностей.

**Как положить текст в папку:**
1. Скопируйте нужный текст в обычный файл. Если текст в PDF, выделите нужные страницы и скопируйте вручную (PDF-парсинг — отдельная тема, мы её обойдём).
2. Сохраните файл в папку `data` под именем `documents.txt`. Убедитесь, что кодировка — **UTF-8** (в Блокноте при сохранении выберите «UTF-8» в выпадающем списке кодировок).
3. Путь к файлу должен быть: `rag_psychology/data/documents.txt`.

**Проверка:** откройте `documents.txt` в VS Code (двойной клик в Explorer). Текст должен отображаться нормально, без «кракозябр».

---

## Шаг 3. Скачайте модель эмбеддингов и установите `numpy`

Эмбеддинги — это числовые векторы, которые представляют смысл текста. Похожие по смыслу фрагменты получают близкие векторы.

1. Откройте **терминал в VS Code**: меню **Terminal → New Terminal** (или нажмите `` Ctrl+` ``). Убедитесь, что в правом верхнем углу выбрано **powershell**.
2. Выполните команду, чтобы скачать модель эмбеддингов `nomic-embed-text` (около 274 МБ). Это одна из самых популярных моделей для RAG, она хорошо работает на коротких и длинных текстах:
   ```powershell
   ollama pull nomic-embed-text
   ```
3. Установите `numpy` для векторных вычислений:
   ```powershell
   pip install numpy
   ```

**Проверка:** выполните `ollama list`. В списке должна появиться строка `nomic-embed-text`.

---

## Шаг 4. Создайте первый скрипт: `build_index.py`

Этот скрипт делает «черновую работу»: читает вашу книгу, режет её на куски (чанки), считает для каждого куска эмбеддинг и сохраняет результаты в два файла. Запускается **один раз** (или заново, если вы поменяли текст книги).

### 4.1. Создайте файл в VS Code

1. В левой панели Explorer наведите курсор на папку `code` — появятся иконки.
2. Нажмите иконку **New File** (лист с плюсом).
3. Введите имя: `build_index.py` и нажмите `Enter`.

### 4.2. Вставьте код

```python
import json
import requests
import numpy as np

# ---------------------------------------------------------------
# НАСТРОЙКИ
# ---------------------------------------------------------------
# Актуальный эндпоинт Ollama для эмбеддингов (ноябрь 2026).
# Старый /api/embeddings считается устаревшим (legacy).
EMBED_URL = "http://localhost:11434/api/embed"
EMBED_MODEL = "nomic-embed-text"

CHUNK_SIZE = 600      # примерно столько символов в одном чанке
OVERLAP = 120         # перекрытие между соседними чанками


def read_documents(filename="../data/documents.txt"):
    """Читает текст из файла и возвращает одну большую строку."""
    with open(filename, "r", encoding="utf-8") as f:
        return f.read()


def split_into_chunks(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """Режет текст на перекрывающиеся куски."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:                      # пропускаем пустые куски
            chunks.append(chunk)
        start = end - overlap          # следующий кусок начинается с перекрытием
    return chunks


def get_embeddings_batch(texts):
    """Отправляет список текстов в Ollama и возвращает список векторов.

    ВАЖНО: используется эндпоинт /api/embed, который принимает
    поле "input" (строка или массив строк) и возвращает поле
    "embeddings" (массив векторов). Это актуальный способ
    на ноябрь 2026 года.
    """
    payload = {
        "model": EMBED_MODEL,
        "input": texts              # передаём сразу несколько текстов
    }
    response = requests.post(EMBED_URL, json=payload, timeout=300)
    response.raise_for_status()
    return response.json()["embeddings"]


def build_and_save_index():
    """Главная функция: читает документы, режет, считает эмбеддинги, сохраняет."""
    text = read_documents()
    chunks = split_into_chunks(text)
    print(f"Получено {len(chunks)} чанков. Считаю эмбеддинги...")

    # Считаем эмбеддинги батчами (по 16 штук), чтобы не перегружать Ollama
    all_vectors = []
    batch_size = 16
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        print(f"  Батч {i // batch_size + 1}: чанки {i + 1}–{i + len(batch)}...")
        vectors = get_embeddings_batch(batch)
        all_vectors.extend(vectors)

    # Сохраняем чанки и векторы в два файла
    with open("chunks.json", "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    np.save("embeddings.npy", np.array(all_vectors))
    print("Готово: chunks.json и embeddings.npy сохранены.")


if __name__ == "__main__":
    build_and_save_index()
```

### 4.3. Что здесь важно понять

- **Почему `/api/embed`, а не `/api/embeddings`?** Официальная документация Ollama на ноябрь 2026 года прямо указывает: старый эндпоинт `/api/embeddings` «has been superseded by /api/embed» (был заменён на `/api/embed`). Новый эндпоинт принимает поле `input` (строку или массив строк) и возвращает поле `embeddings` (массив векторов). Старый принимал `prompt` и возвращал `embedding`. Если скопировать старый код из интернета, он может «тихо» не работать. Мы используем актуальный вариант.
- **Зачем батчи?** Отправлять по одному тексту — медленно. `/api/embed` поддерживает массивы, поэтому мы отправляем по 16 чанков за раз.
- **`split_into_chunks`** — это «нарезка». Перекрытие в 120 символов не даёт разорвать мысль на границе кусков.

**Запустите скрипт.** В терминале VS Code выполните (убедитесь, что вы находитесь в папке `code`; если нет — используйте `cd code`):
```powershell
python build_index.py
```

Дождитесь сообщения «Готово». В папке `code` появятся два файла: `chunks.json` и `embeddings.npy`. Откройте `chunks.json` — убедитесь, что текст нарезан осмысленно, а не посреди слова.

---

## Шаг 5. Создайте второй скрипт: `rag_chat.py`

Этот скрипт загружает сохранённые чанки и векторы, принимает ваш вопрос, находит самые близкие фрагменты и просит модель ответить, опираясь на них.

### 5.1. Создайте файл

В папке `code` создайте новый файл `rag_chat.py` (так же, как создавали `build_index.py`).

### 5.2. Вставьте код

```python
import json
import requests
import numpy as np

# ---------------------------------------------------------------
# НАСТРОЙКИ
# ---------------------------------------------------------------
CHAT_URL = "http://localhost:11434/api/chat"
EMBED_URL = "http://localhost:11434/api/embed"   # актуальный эндпоинт
CHAT_MODEL = "phi3:mini"        # можно заменить на qwen2.5:3b или другую
EMBED_MODEL = "nomic-embed-text"
TOP_K = 3                       # сколько фрагментов подставлять в промпт


def load_index():
    """Загружает чанки и их векторы из файлов."""
    with open("chunks.json", "r", encoding="utf-8") as f:
        chunks = json.load(f)
    vectors = np.load("embeddings.npy")
    return chunks, vectors


def get_embedding(text):
    """Считает эмбеддинг для вопроса пользователя.

    Использует актуальный /api/embed. Поле "input" — строка,
    ответ содержит массив "embeddings" (даже для одной строки),
    поэтому берём первый элемент.
    """
    payload = {"model": EMBED_MODEL, "input": text}
    response = requests.post(EMBED_URL, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["embeddings"][0]


def cosine_similarity(a, b):
    """Косинусное сходство двух векторов: от -1 до 1, чем ближе к 1 — тем похожее."""
    a = np.array(a)
    b = np.array(b)
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def find_relevant_chunks(question, chunks, vectors, top_k=TOP_K):
    """Возвращает top_k чанков, наиболее близких к вопросу."""
    question_vec = get_embedding(question)
    scores = []
    for i, vec in enumerate(vectors):
        sim = cosine_similarity(question_vec, vec)
        scores.append((sim, i))

    scores.sort(reverse=True, key=lambda x: x[0])

    results = []
    for sim, idx in scores[:top_k]:
        results.append((chunks[idx], sim))
    return results


def build_prompt(question, relevant_chunks):
    """Собирает промпт: инструкция + найденные фрагменты + вопрос."""
    context_parts = []
    for i, (chunk, sim) in enumerate(relevant_chunks, start=1):
        context_parts.append(f"[Фрагмент {i}]\n{chunk}")
    context = "\n\n".join(context_parts)

    system_text = (
        "Ты — помощник по психологии. Отвечай на вопрос, опираясь ТОЛЬКО на фрагменты ниже. "
        "Если в фрагментах нет ответа, скажи: «В предоставленных материалах ответа нет». "
        "Не выдумывай факты. По возможности указывай, из какого фрагмента взята мысль."
    )

    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": f"Вопрос: {question}\n\nФрагменты:\n{context}"}
    ]
    return messages


def ask_model(messages):
    """Отправляет промпт в чат-модель и возвращает ответ."""
    payload = {
        "model": CHAT_MODEL,
        "messages": messages,
        "stream": False,
        "options": {"temperature": 0.3}
    }
    response = requests.post(CHAT_URL, json=payload, timeout=300)
    response.raise_for_status()
    return response.json()["message"]["content"]


def main():
    chunks, vectors = load_index()
    print("RAG-чат по психологии. Введите вопрос или 'выход'.")
    print("(После ответа модели будут показаны фрагменты, на которые она опиралась.)\n")

    while True:
        question = input("Вы: ").strip()
        if question.lower() in ("выход", "exit", "quit"):
            print("До свидания!")
            break
        if not question:
            continue

        print("Ищу в документах...")
        try:
            relevant = find_relevant_chunks(question, chunks, vectors)
        except requests.exceptions.ConnectionError:
            print("Ошибка: Ollama не запущена. Проверьте, что программа Ollama работает.")
            continue

        print("\n--- Найденные фрагменты ---")
        for i, (chunk, sim) in enumerate(relevant, start=1):
            preview = chunk[:200].replace("\n", " ")
            print(f"{i}. (сходство {sim:.3f}) {preview}...")

        print("\n--- Ответ модели ---")
        messages = build_prompt(question, relevant)
        try:
            answer = ask_model(messages)
            print(answer)
        except requests.exceptions.ConnectionError:
            print("Ошибка соединения с Ollama.")
        except requests.exceptions.Timeout:
            print("Модель не ответила за отведённое время. Попробуйте ещё раз или возьмите модель меньше.")
        print()


if __name__ == "__main__":
    main()
```

### 5.3. Что здесь важно понять

- **Почему `temperature: 0.3`?** В RAG мы хотим, чтобы модель **опиралась на текст**, а не фантазировала. Низкая температура уменьшает «творчество».
- **Зачем показывать найденные фрагменты?** Это главный инструмент отладки. Если модель ответила плохо, посмотрите, что нашлось. Если нашлось не то — проблема в чанкинге или эмбеддингах, а не в модели.
- **Косинусное сходство** — мера «похожести» двух векторов. Чем ближе к 1, тем семантически ближе тексты. Реализация через `numpy` занимает несколько строк.

---

## Шаг 6. Запустите чат и проверьте

1. Убедитесь, что Ollama работает (значок в трее).
2. В терминале VS Code, находясь в папке `code`, выполните:
   ```powershell
   python rag_chat.py
   ```
3. Задайте 5–7 тестовых вопросов, ответы на которые **точно есть** в вашем тексте. Например, если тема — когнитивные искажения, спросите: «Что такое катастрофизация?» или «Приведи пример сверхобобщения».

**Что должно получиться:** модель отвечает, опираясь на фрагменты, и в ответе видно, что она использует текст, а не выдумывает.

**Если модель говорит «В предоставленных материалах ответа нет»**, хотя ответ там есть:
- Проверьте, какие фрагменты нашлись. Возможно, чанки слишком крупные или ответ «размазан» по нескольким.
- Попробуйте изменить `TOP_K` с 3 на 5.
- Проверьте, что вопрос сформулирован теми же словами, что и текст (эмбеддинги лучше работают, когда лексика пересекается).

---

## Шаг 7. Что сдавать

1. Папка проекта `rag_psychology` с файлами: `data/documents.txt`, `code/build_index.py`, `code/rag_chat.py`, `code/chunks.json`, `code/embeddings.npy`.
2. **`README.md`** (полстраницы):
   - какую тему и какие документы вы взяли;
   - как запустить (`python build_index.py`, затем `python rag_chat.py`);
   - 5 примеров вопросов и ответов (можно скриншотами терминала).
3. **Короткий отчёт** (1 страница):
   - что получилось хорошо;
   - где система ошибалась (нашла не тот фрагмент, ответила «не знаю»);
   - как вы это исправляли.

---

## Шаг 8. Критерии оценки

| Критерий | Вес |
|---|---|
| Программа работает end-to-end (индекс строится, чат отвечает) | 40% |
| На тестовых вопросах находятся релевантные фрагменты | 20% |
| Ответы опираются на фрагменты, модель не «фантазирует» | 20% |
| Понятность кода и README | 10% |
| Рефлексия в отчёте | 10% |

---

## Краткая справка по актуальному API (на 14.11.2026)

- **Эндпоинт эмбеддингов:** `POST http://localhost:11434/api/embed`. Поле запроса — `input` (строка или массив строк). Поле ответа — `embeddings` (массив векторов). Старый `/api/embeddings` считается устаревшим (legacy) и не поддерживает батчи.
- **Модель эмбеддингов:** `nomic-embed-text` (около 274 МБ, 768 измерений) — одна из самых популярных и стабильных для RAG.
- **Косинусное сходство:** стандартная формула `np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))`. Векторы от `/api/embed` уже L2-нормализованы, поэтому косинусное сходство можно считать даже через простое скалярное произведение, но для учебных целей полезно написать формулу полностью.

Если на каком-то шаге что-то пойдёт не так — возвращайтесь к этому шагу и проверяйте внимательно. Проект специально разбит на маленькие шаги, чтобы каждую ошибку можно было найти и исправить.



## Промпт 1. Парсер документов (PDF → текст)

**Зачем:** сейчас вы вручную копируете текст из PDF в `documents.txt`. Это долго и чревато ошибками (переносы, артефакты вёрстки, потерянные абзацы). Промпт просит модель написать отдельный скрипт, который автоматически извлекает текст из PDF и сохраняет его в чистый `.txt`.

**Что важно знать:** для простых (не сканированных) PDF достаточно `pypdf` — это чистый Python-пакет без системных зависимостей, который на ноябрь 2026 года остаётся самым простым и надёжным выбором для базового извлечения текста. Для сканированных PDF (картинок) потребуется OCR, но это выходит за рамки проекта.

**Готовый промпт:**

```text
Ты — терпеливый преподаватель. Помоги мне написать скрипт на Python, который извлекает текст из PDF-файла и сохраняет его в текстовый файл documents.txt для дальнейшего использования в RAG-проекте.

Контекст: у меня есть папка data, в которой лежит книга или лекции в формате PDF (например, data/book.pdf). Мне нужно получить чистый текст без лишних артефактов, чтобы потом нарезать его на чанки.

Требования:
1. Используй библиотеку pypdf (актуальную на ноябрь 2026 года). Никаких PyMuPDF, pdfplumber или OCR — только pypdf для простоты.
2. Скрипт должен называться parse_pdf.py и лежать в папке code.
3. Путь к PDF-файлу задаётся константой в начале скрипта (например, PDF_PATH = "../data/book.pdf").
4. Скрипт должен:
   - открыть PDF через pypdf.PdfReader;
   - пройти по всем страницам и извлечь текст через page.extract_text();
   - склеить текст всех страниц с разделителем "\n\n";
   - сохранить результат в файл "../data/documents.txt" в кодировке UTF-8;
   - вывести в консоль количество страниц и общее количество извлечённых символов.
5. Обработай возможные ошибки:
   - если файл не найден — понятное сообщение;
   - если PDF зашифрован (нужен пароль) — сообщение, что файл защищён;
   - если страница не содержит текста (например, картинка) — пропустить её с предупреждением в консоли.
6. Комментарии в коде — на русском и короткие, для студентов-психологов, знающих только базовый Python.

Покажи полный код parse_pdf.py и кратко объясни, что делает каждая часть. Также напиши команду установки библиотеки в PowerShell.
```

**Что этот промпт тренирует:** работу с файлами, обработку ошибок, понимание того, что PDF — это не просто «текст в коробочке», а структура, из которой текст ещё нужно извлечь.

---

## Промпт 2. Улучшить чанкинг (переход к токенам и рекурсивному разбиению)

**Зачем:** сейчас чанки режутся «по символам» — это грубо. Модель эмбеддингов `nomic-embed-text` работает с токенами, а не с символами. Бенчмарки 2026 года показывают, что **рекурсивное разбиение по 512 токенов с перекрытием 50–100 токенов** — это проверенный дефолт для большинства RAG-приложений, который на реальных документах обгоняет более сложные методы. Для учебного проекта достаточно начать с 512 токенов и перекрытия 10–15%.

**Готовый промпт:**

```text
Перепиши функцию split_into_chunks в моём build_index.py так, чтобы она резала текст не по символам, а по токенам.

Требования:
1. Используй библиотеку tiktoken (pip install tiktoken) с кодировкой cl100k_base — это стандартный токенизатор, который хорошо работает как прокси для большинства моделей. Если tiktoken недоступен, можно использовать простую оценку: 1 токен ≈ 4 символа, но предпочтительнее tiktoken.
2. Размер чанка: 512 токенов. Перекрытие: 64 токена (около 12.5%).
3. Реализуй рекурсивное разбиение: сначала пытайся резать по абзацам ("\n\n"), затем по предложениям ("."), и только в крайнем случае — по токенам. Это сохраняет смысловую целостность кусков.
4. Если после разбиения получается кусок короче 50 токенов, присоедини его к предыдущему (чтобы не было «огрызков»).
5. Функция должна принимать текст и возвращать список строк (чанков).
6. Покажи полный обновлённый build_index.py с этой функцией и краткими комментариями на русском.

Вот мой текущий build_index.py:
[вставьте код build_index.py]
```

**Что этот промпт тренирует:** понимание разницы между символами и токенами; знакомство с рекурсивным разбиением как компромиссом между простотой и качеством.

---

## Промпт 3. Гибридный поиск: добавить BM25 к векторному поиску

**Зачем:** векторный поиск хорошо находит «по смыслу», но иногда промахивается, когда в вопросе есть точный термин или редкое слово. BM25 (классический поиск по ключевым словам) страхует в таких случаях. Исследования 2026 года показывают, что гибридный поиск (BM25 + векторы) с объединением результатов через **Reciprocal Rank Fusion (RRF)** даёт заметный прирост: например, MRR@3 с 0.74 до 0.79, nDCG@3 с 0.76 до 0.81. Это не «магия», а понятная комбинация двух сигналов: лексического и семантического.

**Готовый промпт:**

```text
Улучши мой rag_chat.py: добавь гибридный поиск, который объединяет векторный поиск (косинусное сходство эмбеддингов) и поиск по ключевым словам BM25.

Требования:
1. Используй библиотеку rank_bm25 (pip install rank_bm25) — она простая и не требует тяжёлых зависимостей.
2. При старте программы построй BM25-индекс по всем чанкам (токенизируй текст простым split() по пробелам и приведи к нижнему регистру).
3. Для каждого вопроса:
   - найди top-10 чанков векторным поиском (как сейчас);
   - найди top-10 чанков через BM25;
   - объедини два списка через Reciprocal Rank Fusion (RRF) с параметром k=60 (стандартное значение). RRF: score(chunk) = сумма по каждому списку 1 / (k + rank).
   - выбери top-3 чанка по итоговому RRF-скору.
4. В консоли показывай, какие чанки пришли из векторного поиска, какие из BM25, и итоговый список после RRF — это поможет понять, что происходит.
5. Покажи полный обновлённый rag_chat.py с комментариями на русском.

Вот мой текущий rag_chat.py:
[вставьте код rag_chat.py]
```

**Что этот промпт тренирует:** понимание того, что «поиск» — это не одна волшебная кнопка, а комбинация сигналов; знакомство с RRF как простым и эффективным способом объединения ранжирований.

---

## Промпт 4. Стриминг ответа в терминале

**Зачем:** сейчас `rag_chat.py` ждёт весь ответ целиком и только потом печатает. В первом проекте вы уже делали стриминг для обычного чата — теперь нужно адаптировать его для RAG. Важно: Ollama использует **NDJSON** (Newline-Delimited JSON), а не Server-Sent Events. Каждая строка — самостоятельный JSON-объект, и его нужно разбирать через `json.loads` построчно.

**Готовый промпт:**

```text
Перепиши функцию ask_model и цикл вывода в моём rag_chat.py так, чтобы ответ модели печатался в терминал постепенно (стриминг), а не одним куском.

Требования:
1. Используй эндпоинт /api/chat с параметром stream=True.
2. Ollama присылает NDJSON: каждая строка — отдельный JSON-объект. Разбирай ответ построчно через response.iter_lines():
   - для каждой непустой строки декодируй её (utf-8) и разбери через json.loads;
   - бери поле chunk["message"]["content"] (если оно есть);
   - печатай этот кусочек через print(..., end="", flush=True).
3. После окончания стриминга добавь перевод строки.
4. Обработай ошибки: ConnectionError, HTTPError, Timeout — с понятными сообщениями, как в текущем коде.
5. Сохрани всю текущую логику: поиск фрагментов, показ найденных фрагментов перед ответом.
6. Покажи полный обновлённый rag_chat.py с комментариями на русском.

Вот мой текущий rag_chat.py:
[вставьте код rag_chat.py]
```

**Что этот промпт тренирует:** работу с потоковыми ответами; понимание того, что формат ответа при `stream=True` отличается от обычного JSON.

---

## Промпт 5. Улучшить системный промпт (заставить модель ссылаться на фрагменты)

**Зачем:** даже с RAG модель иногда «забывает» про фрагменты и отвечает «из головы». Системный промпт — главный инструмент управления. Нужно явно потребовать: отвечать только по фрагментам, указывать номер фрагмента, честно говорить «не знаю», если ответа нет.

**Готовый промпт:**

```text
Перепиши системный промпт в функции build_prompt моего rag_chat.py так, чтобы модель строго следовала правилам RAG.

Требования к новому системному промпту:
1. Роль: «Ты — помощник по психологии, который отвечает на вопросы, опираясь ТОЛЬКО на предоставленные фрагменты текста».
2. Обязательное правило: «Если в фрагментах нет прямого ответа на вопрос, скажи: "В предоставленных материалах ответа нет" и не пытайся угадать».
3. Требование ссылаться на источник: «В конце каждого утверждения указывай номер фрагмента в квадратных скобках, например: [Фрагмент 2]».
4. Запрет на выдумывание: «Не добавляй факты, которых нет во фрагментах, даже если ты уверен, что они верны».
5. Краткость: «Отвечай по существу, без вступлений вроде "Конечно!" или "Отличный вопрос!"».

Покажи полный обновлённый rag_chat.py с новым системным промптом. Комментарии — на русском.

Вот мой текущий rag_chat.py:
[вставьте код rag_chat.py]
```

**Что этот промпт тренирует:** понимание того, что системный промпт — это «инструкция по эксплуатации» модели, а не просто вежливая фраза.

---

## Промпт 6. Добавить порог отсечения (не подставлять «мусор»)

**Зачем:** сейчас в промпт всегда попадают top-3 чанка, даже если они совсем нерелевантны. Если вопрос не по теме документов, модель получит «мусор» и может ответить ерунду. Решение — ввести порог косинусного сходства: если лучший чанк ниже порога, не подставлять его вообще и честно сказать пользователю, что в материалах нет ответа.

**Готовый промпт:**

```text
Добавь в rag_chat.py порог отсечения для найденных фрагментов.

Требования:
1. Задай константу MIN_SIMILARITY = 0.35 в начале файла (подбери экспериментально: для nomic-embed-text хорошие результаты обычно имеют сходство 0.5–0.7, «мусор» — ниже 0.3).
2. В функции find_relevant_chunks после сортировки отбрось все чанки, у которых сходство ниже MIN_SIMILARITY.
3. Если после отсечения не осталось ни одного чанка, в main() не отправляй запрос в модель, а выведи: «В предоставленных материалах нет информации по этому вопросу. Попробуйте переформулировать или задать другой вопрос».
4. Если осталось меньше TOP_K чанков — используй столько, сколько есть.
5. Покажи полный обновлённый rag_chat.py с комментариями на русском.

Вот мой текущий rag_chat.py:
[вставьте код rag_chat.py]
```

**Что этот промпт тренирует:** понимание того, что RAG — это не «всегда отвечай», а «отвечай, когда есть на что опереться».

---

## Промпт 7. Сохранение и загрузка индекса без пересчёта

**Зачем:** сейчас `build_index.py` при каждом запуске пересчитывает эмбеддинги для всех чанков. Если документы не менялись, это трата времени. Нужно проверять, существует ли уже готовый индекс, и загружать его.

**Готовый промпт:**

```text
Улучши build_index.py так, чтобы он не пересчитывал эмбеддинги, если индекс уже существует и документы не менялись.

Требования:
1. При запуске проверяй, существуют ли файлы chunks.json и embeddings.npy.
2. Если оба файла существуют, выведи: «Индекс уже построен. Если вы изменили документы, удалите chunks.json и embeddings.npy и запустите снова». И заверши работу без пересчёта.
3. Добавь флаг --force (или переменную FORCE_REBUILD = False в начале файла), который позволяет принудительно перестроить индекс, даже если файлы есть.
4. Покажи полный обновлённый build_index.py с комментариями.

Вот мой текущий build_index.py:
[вставьте код build_index.py]
```

**Что этот промпт тренирует:** понимание жизненного цикла индекса; аккуратность при работе с файлами.

---

## Промпт 8. Автоматическая оценка качества поиска

**Зачем:** чтобы понять, стало ли лучше после изменений, нужны метрики. Простейший способ — задать 10 тестовых вопросов с заранее известными правильными ответами (или хотя бы «правильными» фрагментами) и посчитать, на каком месте (rank) находится нужный фрагмент. Это учит не «верить на глаз», а измерять.

**Готовый промпт:**

```text
Добавь в проект отдельный скрипт evaluate.py, который оценивает качество поиска на наборе тестовых вопросов.

Требования:
1. Создай файл test_questions.json со списком из 10 вопросов. Для каждого вопроса укажи:
   - question: текст вопроса;
   - expected_keywords: список из 1–3 ключевых слов, которые должны быть в правильном чанке (например, для вопроса «Что такое катастрофизация?» — ["катастрофизация", "преувеличение"]).
2. Скрипт evaluate.py:
   - загружает chunks.json, embeddings.npy и test_questions.json;
   - для каждого вопроса находит top-5 чанков (тем же способом, что в rag_chat.py);
   - проверяет, содержит ли хотя бы один из top-5 чанков все expected_keywords;
   - считает, на каком месте (1–5) впервые встречается «правильный» чанк;
   - выводит таблицу: вопрос | найден ли правильный чанк | на каком месте;
   - в конце выводит итоговую метрику: долю вопросов, где правильный чанк попал в top-3 (это простой аналог Recall@3).
3. Используй только стандартные библиотеки + numpy + requests (как в остальном проекте).
4. Покажи полный код evaluate.py и пример test_questions.json с 3 вопросами для демонстрации.

Контекст: вот мой текущий find_relevant_chunks из rag_chat.py:
[вставьте функцию find_relevant_chunks или весь rag_chat.py]
```

**Что этот промпт тренирует:** мышление в терминах метрик; понимание того, что «улучшение» — это не субъективное ощущение, а измеримая величина.

---

## Как пользоваться этими промптами (памятка)

1. **Выберите один промпт**, скопируйте его в DeepSeek или ChatGPT.
2. **Приложите свой текущий код** (полный файл или указанную функцию).
3. Получите обновлённый код. **Сравните** с исходным: что именно изменилось? Прочитайте комментарии.
4. **Запустите** и проверьте, что улучшение действительно работает (например, для промпта 8 — сравните метрику до и после).
5. **Сохраните копию** перед изменениями: `rag_chat_v1.py`, `rag_chat_v2.py` — чтобы можно было откатиться.

Эти промпты покрывают основные «болевые точки» RAG-проекта: качество парсинга, качество чанкинга, качество поиска, честность модели и измеримость результата. Начинать логично с **Промпта 1** (парсер), затем **Промпта 2** (чанкинг), затем **Промпта 5** (системный промпт) — это даст наибольший эффект при минимальных изменениях.